# Direk Kaggle Worker — gear 253B (Qwen 2.5 3B)

T3 heavy batch worker ng MOVIE CREATOR. Nagse-serve ito ng screenplay endpoints
sa GPU ng Kaggle **o Colab**, sa likod ng Cloudflare quick tunnel.

## Ano ang inaayos dito

1. **`device_map="auto"` sa T4 x2** — hinahati ang 3B model sa dalawang GPU.
   3B sa fp16 ay ~6.2 GB; kasya sa isang T4 (15 GB). Nakapin na ngayon sa isang
   GPU, at may fallback kung ma-out of memory.
2. **Hardcoded na CGNAT IP** — `100.122.147.65` ay nasa `100.64.0.0/10`
   (Tailscale/CGNAT). **Hindi kayang abutin ng Kaggle iyon**, kaya laging
   pumapalya ang sync nang tahimik. Ngayon: configurable, at **sinusubukan
   muna bago mag-claim ng tagumpay**.
3. **`/breakdown_scenes` ay walang error path** — 500 ang sagot kapag nabigo
   ang generation, imbes na ang honest na 503 ng ibang endpoints.
4. **Isang notebook, dalawang host** — Kaggle at Colab.

## Honest na patakaran

Walang pekeng script. Kapag hindi pa handa ang model, **503 `AI_UNAVAILABLE`**
ang sagot at ang app ang magpapasya (lilipat sa lokal na Ollama). Ang
`search_titles` ay galing sa **totoong** Kaggle dataset.

## ⚠️ Hindi ito gumagawa ng video

Text lang ito: script, character bible, scene breakdown. Ang video ay hiwalay
na notebook (`kaggle_colab_fusion_studio.ipynb`). Huwag asahan ang video dito.

## Paano gamitin

1. Kaggle: **Settings -> Accelerator -> GPU T4 x2**, **Internet -> On**.
   Colab: **Runtime -> Change runtime type -> T4 GPU**.
2. **Run all**. Hintayin ang cell 8 — doon lalabas ang `KAGGLE_PILOT_URL`.
3. Kopyahin ang URL sa Server-2 `.env` bilang `KAGGLE_PILOT_URL`, tapos
   i-restart ang app. Kung naka-configure ang sync, awtomatiko itong
   magpapadala — at sasabihin nang tapat kung nabigo.
4. Ang tunnel URL ay **nagbabago bawat session**.

In [ ]:
#@title 1. Host, disk at config
import os, shutil, subprocess, sys

# Which host is this? Everything downstream keys off these values.
IS_KAGGLE = os.path.isdir("/kaggle")
IS_COLAB = not IS_KAGGLE and os.path.isdir("/content")
BASE = "/kaggle/working" if IS_KAGGLE else ("/content" if IS_COLAB else os.getcwd())
HOST = "Kaggle" if IS_KAGGLE else ("Colab" if IS_COLAB else "Local")

os.makedirs(BASE, exist_ok=True)

# One GPU on purpose. A 3B model in fp16 is ~6.2 GB and fits on a single T4;
# device_map="auto" spread it over both cards and only added transfer overhead.
MODEL_ID = os.environ.get("GEAR_MODEL_ID", "Qwen/Qwen2.5-3B-Instruct")
GEAR_DEVICE = os.environ.get("GEAR_DEVICE", "cuda:0")

# The sync target must be reachable from the public internet. A Tailscale or
# RFC1918 address will never work from a Kaggle container, so it is rejected
# here instead of failing silently later.
LROS_SERVER_URL = os.environ.get("LROS_SERVER_URL", "").strip()


def secret(name):
    """Read a secret from the host's own store. Never hardcode one."""
    try:
        if IS_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
        if IS_COLAB:
            from google.colab import userdata
            return userdata.get(name)
    except Exception:
        return None
    return None


if not LROS_SERVER_URL:
    LROS_SERVER_URL = (secret("LROS_SERVER_URL") or "").strip()

print(f"Host  : {HOST}")
print(f"BASE  : {BASE}")
print(f"Model : {MODEL_ID}")
print(f"Sync  : {LROS_SERVER_URL or '(wala — manual na pag-copy ng URL)'}")

In [ ]:
#@title 2. GPU report
# Report the GPU we actually got, so a CPU-only session is obvious immediately.
smi = subprocess.run(
    "nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
    shell=True, capture_output=True, text=True,
)
if smi.returncode != 0 or not smi.stdout.strip():
    sys.exit(
        "WALANG GPU.\n"
        + ("Kaggle: Settings -> Accelerator -> GPU T4 x2, tapos Run all muli."
           if IS_KAGGLE else
           "Colab: Runtime -> Change runtime type -> T4 GPU, tapos Run all muli.")
    )

GPUS = [line.strip() for line in smi.stdout.strip().splitlines() if line.strip()]
print(f"GPUs na nakita: {len(GPUS)}")
for i, g in enumerate(GPUS):
    print(f"  [{i}] {g}")
print(f"Gagamitin: {GEAR_DEVICE} (isang GPU lang — sapat ang 15 GB sa 3B fp16)")

free = shutil.disk_usage(BASE).free / 1e9
print(f"Free disk sa {BASE}: {free:.1f} GB")

In [ ]:
#@title 3. Install
# transformers 5.x declares `torch>=2.6`. Kaggle's image ships an older torch,
# so an unpinned `-U transformers` drags a CUDA torch upgrade behind it and that
# is what breaks the session. Pin below 5: 4.57 already accepts `dtype=` and
# still needs only torch>=2.2, so the preinstalled torch is left alone.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U",
     "transformers>=4.56,<5", "accelerate", "sentencepiece", "requests"],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "fastapi", "uvicorn", "pydantic"],
    check=True,
)
import transformers, torch
print(f"transformers {transformers.__version__} | torch {torch.__version__}")
if int(transformers.__version__.split(".")[0]) >= 5:
    raise RuntimeError(
        "transformers 5.x ang na-install at nangangailangan iyon ng torch>=2.6. "
        "I-restart ang session, tapos Run all muli."
    )
print("installed")


In [ ]:
#@title 4. Load the model on ONE GPU
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_ID)

if torch.cuda.is_available():
    torch.cuda.set_device(int(GEAR_DEVICE.split(":")[-1]))
    dtype = torch.float16
else:
    dtype = torch.float32

try:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=dtype,
        device_map={"": GEAR_DEVICE} if torch.cuda.is_available() else None,
        low_cpu_mem_usage=True,
    )
except (TypeError, ValueError):
    # transformers < 4.56 has no `dtype` keyword.
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=dtype,
        device_map={"": GEAR_DEVICE} if torch.cuda.is_available() else None,
        low_cpu_mem_usage=True,
    )
except torch.cuda.OutOfMemoryError:
    sys.exit(
        "Kulang ang VRAM para sa " + MODEL_ID + " sa " + GEAR_DEVICE + ".\n"
        "Subukan ang GEAR_MODEL_ID=Qwen/Qwen2.5-1.5B-Instruct."
    )

model.eval()
MODEL_READY = True

if torch.cuda.is_available():
    used = torch.cuda.memory_allocated(0) / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Model loaded sa {next(model.parameters()).device} — {used:.1f} GB / {total:.1f} GB")
else:
    print("Model loaded sa CPU (mabagal — walang GPU)")

In [ ]:
#@title 5. Generation helper + Direk persona
def gen(system: str, user: str, max_tokens: int = 1200, temperature: float = 0.7) -> str:
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=temperature > 0,
            temperature=temperature if temperature > 0 else None,
            top_p=0.9,
        )
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


SKILL_LIST = """- write_script: Sumulat ng logline, synopsis, script, dialogue, scene list.
- shot_list: Gumawa ng shot list at storyboard prompts bawat eksena.
- character_bible: Gumawa ng karakter, itsura, ugali, boses (consistent sa buong pelikula).
- ai_video_prompt: Isulat ang image-to-video / text-to-video prompt bawat shot."""

RULES = """PATAKARAN SA SAGOT (sundin nang mahigpit):
1. Sumagot sa Filipino/Taglish, kahit English ang tanong.
2. Kung kulang ang detalye, MAGTANONG LANG muna. Huwag mag-imbento ng genre, haba, o budget.
3. Kung totoong tao ang gustong gawing bida o quote, TUMANGGI nang maikli at mag-alok ng fiktional na karakter."""

DIREK_SYSTEM = (
    "Ikaw si Direk, AI Movie Director at Creator ng boss mong si Rex.\n"
    "Trabaho mo: gumawa ng pelikula/serye/ad mula ideya hanggang final, sa pinakamababang gastos pero maayos na kalidad.\n\n"
    "ASAL:\n"
    "- Magalang, masigla, diretso. Maikling sagot. Filipino (Taglish OK).\n"
    "- Gumamit ng pipeline: IDEYA -> SCRIPT -> KARAKTER -> SHOT LIST -> VISUAL -> EDIT -> PUBLISH.\n"
    "- Unahin ang LIBRENG paraan. Sabihin kung may gastos ang isang hakbang.\n"
    "- Huwag mag-imbento ng katotohanan o pangalan ng totoong tao.\n\n"
    "MGA SKILL MO (lokal lang, gawin mo mismo sa sagot):\n" + SKILL_LIST + "\n\n" + RULES
)
print("gen() ready")

In [ ]:
#@title 6. FastAPI worker (app contract + honest 503)
import json, threading, time
from fastapi import FastAPI
from fastapi.responses import JSONResponse
from pydantic import BaseModel
import uvicorn

class BriefReq(BaseModel):
    brief: str

class ScriptReq(BaseModel):
    script: str

app = FastAPI()


def unavailable(reason: str):
    # Honest 503: ginagawa itong PILOT_UNAVAILABLE ng app, hindi pekeng script.
    return JSONResponse(
        {"error": reason, "code": "AI_UNAVAILABLE", "model": MODEL_ID},
        status_code=503,
    )


@app.get("/health")
def health():
    return {"ok": True, "model": MODEL_ID, "persona": "Direk", "model_ready": MODEL_READY}


@app.get("/v1/models")
def models():
    return {"object": "list", "data": [{"id": MODEL_ID, "object": "model"}]}


@app.post("/generate_script")
def generate_script(req: BriefReq):
    if not req.brief.strip():
        return JSONResponse({"error": "Walang brief."}, status_code=400)
    try:
        text = gen(
            DIREK_SYSTEM,
            f"Brief: {req.brief}\n\nIsulat ang buong screenplay: pamagat, logline, at bawat eksena.",
            max_tokens=1200, temperature=0.7,
        )
        return {"script": text, "model": MODEL_ID}
    except Exception as exc:
        return unavailable(f"Nabigo ang script generation: {exc}")


@app.post("/generate_character_bible")
def generate_character_bible(req: ScriptReq):
    if not req.script.strip():
        return JSONResponse({"error": "Walang script."}, status_code=400)
    try:
        text = gen(
            DIREK_SYSTEM,
            f"Ito ang screenplay:\n\n{req.script}\n\nIsulat ang character bible: pangalan, itsura, ugali, boses ng bawat karakter.",
            max_tokens=1000, temperature=0.6,
        )
        return {"bible": text, "model": MODEL_ID}
    except Exception as exc:
        return unavailable(f"Nabigo ang character bible: {exc}")


@app.post("/breakdown_scenes")
def breakdown_scenes(req: ScriptReq):
    if not req.script.strip():
        return JSONResponse({"error": "Walang script."}, status_code=400)
    try:
        raw = gen(
            'Magbalik lamang ng JSON: {"scenes":[{"heading":"","narration":"","visual":"","mood":""}]}. Walang ibang teksto.',
            f"Hatiin sa 18 eksena ang screenplay na ito:\n\n{req.script}",
            max_tokens=2000, temperature=0.3,
        )
    except Exception as exc:
        # Dati ay 500 dito — hindi pareho sa ibang endpoints. Ngayon 503 din,
        # para ang app ang lilipat sa lokal na Ollama sa halip na pumalya.
        return unavailable(f"Nabigo ang scene breakdown: {exc}")

    s, e = raw.find("{"), raw.rfind("}")
    try:
        return json.loads(raw[s:e + 1])
    except Exception:
        # Hindi valid JSON: ibalik ang teksto, walang pekeng breakdown.
        return {"text": raw}


def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")


threading.Thread(target=run_server, daemon=True).start()
print("worker starting sa 127.0.0.1:8000")

In [ ]:
#@title 7. Wait until the worker is REALLY ready
# A fixed sleep() lied: it said "up" even when the port was not listening yet.
# Poll /health instead, so readiness is a fact and not a guess.
import requests

HEALTH = "http://127.0.0.1:8000/health"
ready = False
for attempt in range(60):
    try:
        r = requests.get(HEALTH, timeout=2)
        if r.status_code == 200 and r.json().get("model_ready"):
            ready = True
            print(f"Worker UP pagkalipas ng {attempt + 1}s:", r.json())
            break
    except Exception:
        pass
    time.sleep(1)

if not ready:
    raise RuntimeError("Hindi umandar ang worker sa loob ng 60s. Tingnan ang logs sa itaas.")

In [ ]:
#@title 8. Cloudflare tunnel (named kung may token, quick kung wala)
# Ang quick tunnel ay may bagong URL kada run, kaya kailangang i-paste muli sa
# app. Kung may CF_TUNNEL_TOKEN (Kaggle/Colab secret), gamitin ang named tunnel:
# permanente ang hostname, kaya isang beses lang ang KAGGLE_PILOT_URL.
import re

CF = os.path.join(BASE, "cloudflared")
if not os.path.exists(CF):
    subprocess.run(
        ["wget", "-q", "-O", CF,
         "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"],
        check=True,
    )
    os.chmod(CF, 0o755)

CF_TUNNEL_TOKEN = (secret("CF_TUNNEL_TOKEN") or "").strip()
CF_PUBLIC_URL = (secret("CF_PUBLIC_URL") or "").strip()

LOG = os.path.join(BASE, "cf.log")
PUBLIC_URL = None
cf_proc = None

if CF_TUNNEL_TOKEN:
    print("CF_TUNNEL_TOKEN: meron — named tunnel ang gagamitin")
    # The token goes in through TUNNEL_TOKEN, never argv, so `ps` cannot read it.
    tun_env = os.environ.copy()
    tun_env["TUNNEL_TOKEN"] = CF_TUNNEL_TOKEN
    cf_proc = subprocess.Popen(
        [CF, "tunnel", "--no-autoupdate", "run", "--url", "http://127.0.0.1:8000"],
        stdout=open(LOG, "w"), stderr=subprocess.STDOUT, env=tun_env,
    )
    # A named tunnel keeps a fixed hostname that the token does not reveal, so it
    # has to come from a secret.
    if CF_PUBLIC_URL:
        public = CF_PUBLIC_URL.strip().rstrip("/")
        PUBLIC_URL = public if public.startswith("http") else "https://" + public
    else:
        for _ in range(120):
            time.sleep(1)
            try:
                with open(LOG) as f:
                    m = re.search(r"https://(?!.*trycloudflare)[a-z0-9.-]+\.[a-z]{2,}", f.read())
            except FileNotFoundError:
                m = None
            if m:
                PUBLIC_URL = m.group(0)
                break
else:
    print("CF_TUNNEL_TOKEN: wala — quick tunnel (bagong URL kada run)")
    cf_proc = subprocess.Popen(
        [CF, "tunnel", "--url", "http://127.0.0.1:8000"],
        stdout=open(LOG, "w"), stderr=subprocess.STDOUT,
    )
    for _ in range(120):  # hanggang 2 minuto
        time.sleep(1)
        try:
            with open(LOG) as f:
                m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", f.read())
        except FileNotFoundError:
            m = None
        if m:
            PUBLIC_URL = m.group(0)
            break

if PUBLIC_URL is None:
    hint = ""
    if CF_TUNNEL_TOKEN and not CF_PUBLIC_URL:
        hint = ("\nWalang hostname sa log. I-set din ang CF_PUBLIC_URL "
                "(hal. https://pilot.example.com) sa Secrets.")
    raise RuntimeError(f"Walang tunnel URL pagkalipas ng 2 minuto. Tingnan ang {LOG}{hint}")

with open(os.path.join(BASE, "tunnel_url.txt"), "w") as f:
    f.write(PUBLIC_URL)
print("KAGGLE_PILOT_URL:", PUBLIC_URL)


In [ ]:
#@title 9. Report URL + honest sync
# The old cell posted to a hardcoded 100.x address that Kaggle can never reach,
# and swallowed the error. This one refuses such an address up front and reports
# the real HTTP status.
import ipaddress, urllib.parse

UPDATE_ENDPOINT = "/api/pilot/update-kaggle-url"


def sync_problem(url: str):
    """Return why a sync target is unreachable, or None if it looks usable."""
    host = urllib.parse.urlparse(url).hostname or ""
    if not host:
        return "Walang hostname sa LROS_SERVER_URL."
    try:
        ip = ipaddress.ip_address(host)
    except ValueError:
        return None  # A DNS name may still resolve to something public.
    if ip.is_loopback:
        return f"{host} ay loopback — hindi kayang abutin mula sa Kaggle."
    # 100.64.0.0/10 is CGNAT (Tailscale). ipaddress reports is_private=False for
    # it, so without this check a Tailscale address slips through as "public".
    if ip in ipaddress.ip_network("100.64.0.0/10"):
        return f"{host} ay CGNAT/Tailscale — hindi kayang abutin mula sa Kaggle."
    if ip.is_private:
        return f"{host} ay private — hindi kayang abutin mula sa Kaggle."
    return None


print("=" * 60)
print("KAGGLE_PILOT_URL:", PUBLIC_URL)
print("=" * 60)
print("Ilagay ito sa Server-2 .env, tapos i-restart ang app:")
print(f"KAGGLE_PILOT_URL={PUBLIC_URL}")
print("=" * 60)

if not LROS_SERVER_URL:
    print("\nWalang LROS_SERVER_URL — laktawan ang auto-sync.")
    print("I-set ito sa Kaggle Secret (Add-ons -> Secrets) para awtomatiko.")
else:
    problem = sync_problem(LROS_SERVER_URL)
    if problem:
        print(f"\nHINDI nag-sync: {problem}")
        print("Kailangan ng PUBLIC na URL ng Server-2 (hal. Cloudflare tunnel).")
        print("Manu-manong kopyahin ang KAGGLE_PILOT_URL sa itaas.")
    else:
        # The app keeps APP_AUTH_TOKEN private: it unlocks every billable
        # endpoint, so it must never be pasted into a notebook. Auto-sync uses
        # its own narrow secret, and the route answers 404 when the app has no
        # PILOT_SYNC_TOKEN set.
        sync_token = (secret("PILOT_SYNC_TOKEN") or "").strip()
        if not sync_token:
            print("\nWalang PILOT_SYNC_TOKEN sa Secrets — laktawan ang auto-sync.")
            print("Manu-manong kopyahin ang KAGGLE_PILOT_URL sa itaas.")
        else:
            try:
                res = requests.post(
                    f"{LROS_SERVER_URL.rstrip('/')}{UPDATE_ENDPOINT}",
                    json={"kaggle_url": PUBLIC_URL},
                    headers={"Authorization": f"Bearer {sync_token}"},
                    timeout=15,
                )
                if 200 <= res.status_code < 300:
                    print(f"\nSync OK ({res.status_code}) — natanggap ng Server-2 ang bagong URL.")
                    # The app reports whether the URL survives a restart. Without
                    # Firestore the host has no persistent disk, so the URL is
                    # lost on the next spin-down and must be pasted again.
                    try:
                        durable = res.json().get("durable")
                    except Exception:
                        durable = None
                    if durable is True:
                        print("PERMANENTE: naka-save sa Firestore, gagana pa rin pagkatapos mag-restart.")
                    elif durable is False:
                        print("PANSAMANTALA: walang Firestore/disk sa server —")
                        print("mawawala ang URL kapag nag-restart. I-paste muli mamaya,")
                        print("o i-set ang FIREBASE_SERVICE_ACCOUNT para permanente.")
                else:
                    print(f"\nSync NABIGO ({res.status_code}): {res.text[:200]}")
                    print("Manu-manong kopyahin ang URL sa itaas.")
            except Exception as exc:
                print(f"\nSync NABIGO: {exc}")
                print("Manu-manong kopyahin ang URL sa itaas.")


In [ ]:
#@title 10. Smoke test (real endpoints + 400 guard)
# Tests the real app contract, and proves the honest 503 path too.
import requests

assert PUBLIC_URL, "Patakbuhin muna ang tunnel cell (8)."

h = requests.get(PUBLIC_URL + "/health", timeout=30)
print("health:", h.status_code, h.json())

r = requests.post(
    PUBLIC_URL + "/generate_script",
    json={"brief": "Drama tungkol sa mangingisda sa Navotas."},
    timeout=300,
)
print("generate_script:", r.status_code)
print(str(r.json().get("script"))[:300])

b = requests.post(
    PUBLIC_URL + "/generate_character_bible",
    json={"script": "Isang mangingisda ang nawalan ng bangka."},
    timeout=300,
)
print("generate_character_bible:", b.status_code)

sc = requests.post(
    PUBLIC_URL + "/breakdown_scenes",
    json={"script": "Isang mangingisda ang nawalan ng bangka."},
    timeout=300,
)
print("breakdown_scenes:", sc.status_code, list(sc.json().keys()))

# Empty input must be a 400, not a generated script.
bad = requests.post(PUBLIC_URL + "/generate_script", json={"brief": "  "}, timeout=30)
print("empty brief ->", bad.status_code, "(dapat 400)")
assert bad.status_code == 400, "Hindi tumatanggi sa walang laman na brief"
print("\nSMOKE TEST PASSED")

In [ ]:
#@title 11. Direk training seed (direk_train.jsonl)
# LoRA seed data. Every pair here follows the RULES: ask first, refuse real
# people, never invent a budget or a quote.
import json

examples = [
    ("Gusto kong gumawa ng pelikula tungkol sa mangingisda.",
     "Magandang ideya! Bago ako magsimula, tatlong tanong muna: ilang minuto ang gusto mo, anong mood (masaya, malungkot, o nakakakakaba), at may budget ka ba?"),
    ("Sumulat ka ng logline para sa thriller sa lumang bahay sa QC.",
     "Sige. Bago ang logline, kailangan ko munang malaman: gaano katagal ang video, at supernatural o sikolohikal ang takot na gusto mo?"),
    ("Gawin mo si Manny Pacquiao bilang bida at sabihin ang totoong quote niya.",
     "Hindi ako gumagawa ng quote o diyalogo para sa totoong tao. Puwede nating gawing kathang-isip na boksingero ang bida. Anong klase ng kwento: inspirasyon, drama, o komedya?"),
    ("Magkano ang gastos para sa AI video?",
     "Depende sa serbisyo. Libre ang script, shot list, at character bible na gawin ko. Ang AI video generation ay karaniwang may bayad. Ilang segundo ang kailangan mo?"),
    ("Gumawa ka ng script para sa maikling pelikula.",
     "Puwede. Tanong muna: anong genre, ilang minuto, at sino ang pangunahing karakter? Kapag nakuha ko na ang tatlong iyon, sisimulan ko ang eksena 1."),
    ("Anong genre ang maganda para sa maikling pelikula?",
     "Depende sa budget at kwento mo. Para sa mababang gastos, maganda ang drama at thriller na iisang lokasyon lang. Anong mood ang gusto mong maramdaman ng manonood?"),
    ("Gawin mo si Jose Rizal bilang bida.",
     "Hindi ako gagawa ng kwento na gumagamit ng totoong tao bilang bida nang walang pahintulot. Puwede nating gawing kathang-isip na iskolar sa Maynila ang bida. Gusto mo bang ituloy?"),
    ("Ilang eksena ang kailangan para sa 3 minutong video?",
     "Karaniwang 4 hanggang 6 na eksena para sa 3 minuto. Gusto mo bang ako na ang maghati ng eksena base sa kwento mo?"),
    ("Gusto ko ng pelikula, ikaw na bahala.",
     "Sige, pero kailangan ko ng kahit kaunting detalye para hindi ako manghula: anong genre, gaano katagal, at anong mood? Kahit isa lang ay sapat na para makapagsimula."),
    ("Isulat mo ang dialogue ng isang sikat na artista.",
     "Hindi ako sumusulat ng dialogue na ipinapasa bilang sinabi ng totoong artista. Puwede kong isulat ang dialogue ng kathang-isip na karakter sa kwento mo. Ano ang tagpo?"),
    ("Ano ang unang hakbang sa paggawa ng pelikula?",
     "Simulan sa ideya at isang pangungusap na buod. Susunod ang script, tapos ang karakter at shot list. Gusto mo bang simulan natin sa ideya mo?"),
    ("Kailan ang pinakamagandang oras para mag-post sa social media?",
     "Hindi ko kayang sabihin nang sigurado nang hindi tinitingnan ang datos ng audience mo. Puwede tayong gumamit ng growth tool sa susunod na yugto para dito."),
]

out = os.path.join(BASE, "direk_train.jsonl")
with open(out, "w", encoding="utf-8") as f:
    for user, assistant in examples:
        f.write(json.dumps({"messages": [
            {"role": "user", "content": user},
            {"role": "assistant", "content": assistant},
        ]}, ensure_ascii=False) + "\n")

print("Nakasulat:", len(examples), "halimbawa sa", out)

In [ ]:
#@title 12. Grounded title lookup
# Grounded lookup from the real Kaggle dataset. No invented titles.
import glob
import pandas as pd

cands = glob.glob("/kaggle/input/**/netflix_titles.csv", recursive=True)
if not cands:
    print("Walang netflix_titles.csv sa /kaggle/input — laktawan ang grounding.")
    _df = None
else:
    NETFLIX_PATH = cands[0]
    _df = pd.read_csv(
        NETFLIX_PATH,
        usecols=["title", "type", "listed_in", "release_year", "duration"],
    )
    print("Dataset:", NETFLIX_PATH, "| rows:", len(_df))

GENRE_MAP = {
    "action": "action", "aksyon": "action",
    "drama": "drama",
    "horror": "horror", "katatakutan": "horror",
    "sci-fi": "sci-fi", "sci fi": "sci-fi", "science fiction": "sci-fi",
    "romance": "romantic", "romantic": "romantic", "romantiko": "romantic",
    "love story": "romantic",
}


def find_genre(text):
    t = text.lower()
    for word, genre in GENRE_MAP.items():
        if word in t:
            return genre
    return None


def search_titles(genre, kind=None, n=5):
    if _df is None:
        return []
    n = min(int(n), 5)
    d = _df
    if kind:
        d = d[d["type"].str.lower() == kind.lower()]
    d = d[d["listed_in"].fillna("").str.lower().str.contains(genre.lower())]
    return d.head(n).to_dict(orient="records")


for t in ["Gumawa ka ng action na pelikula", "Anong drama ang maganda?",
          "Gusto ko ng horror, 3 minuto", "Sci-fi na kwento tungkol sa Mars",
          "Gusto ko ng romance"]:
    g = find_genre(t)
    res = search_titles(g, kind="Movie", n=5) if g else []
    print(f"{t!r} -> genre: {g}, resulta: {len(res)}")

In [ ]:
#@title 13. Secrets check (values never printed)
# Reads the three secrets the project uses and reports only whether each one is
# present. Nothing is printed, logged, or written to disk, so this is safe to
# run in a notebook that might be shared.
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

# CF_TUNNEL_TOKEN -> read by cell 8 through secret() to start a named tunnel.
# GEMINI_API_KEY  -> belongs to the app (Render), not to this worker.
# OPEN_MIND_SECRET-> not read anywhere in Rex217-0/MOVIES as of this commit.
WANTED = ("CF_TUNNEL_TOKEN", "GEMINI_API_KEY", "OPEN_MIND_SECRET")

for name in WANTED:
    try:
        value = user_secrets.get_secret(name)
    except Exception as exc:
        # A missing secret raises rather than returning None, and the message
        # can echo the name, so report only the type.
        print(f"{name:<18} : WALA ({type(exc).__name__})")
        continue
    print(f"{name:<18} : {'MERON' if value else 'WALA'} (len={len(value or '')})")

print()
print("Tandaan: ang cell 8 ay gumagamit ng CF_TUNNEL_TOKEN para sa named tunnel.")
print("Ang GEMINI_API_KEY ay sa app, hindi sa notebook na ito.")
